In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!nvidia-smi

Sun Apr 19 19:08:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:4A:00.0 Off |                    0 |
| N/A   28C    P8             33W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Put project root on sys.path so "source" is importable
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> root/
PROJECT_ROOT = PROJECT_ROOT #/ "lcms-foundation-model"
print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

/home/mpominova/lcms-foundation-model


In [4]:
import os
import yaml
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import pytorch_lightning as L
from tqdm import tqdm
from depthcharge.data import SpectrumDataset, spectra_to_df, preprocessing
from torch.utils.data import DataLoader

from source.dataset import LanceMapDataset, RunDataset
from source.model import MS1Encoder
from source.scheduler import CosineWarmupScheduler
from source.config import ExperimentConfig, DataConfig, ModelConfig, OptimizerConfig, TrainingConfig

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# we might want to add this to the MS1Encoder for some debugging / control
# but later / optional

# # Display prediction sample
# n = 30
# mz_bins_true, I_true = target_mz_bins[:n].cpu().numpy(), target_I[:n].cpu().numpy()
# mz_bins_pred, I_pred = pred_mz_bins[:n].argmax(dim=1).cpu().numpy(), pred_I[:n].cpu().numpy()
# sample_df = np.column_stack((
#     mz_bins_true.ravel(), 
#     I_true.ravel(), 
#     mz_bins_pred.ravel(), 
#     # I_pred.ravel()
# ))
# sample_df = pd.DataFrame(
#     sample_df, columns=[
#         "mz_bins_true", 
#         "I_true", 
#         "mz_bins_pred", 
#         # "I_pred"
#     ]
# )
# display(sample_df)

In [6]:
def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config_dict = yaml.safe_load(f)
        config = ExperimentConfig(
            name=config_dict['name'],
            data=DataConfig(**config_dict['data']),
            model=ModelConfig(**config_dict['model']),
            optimizer=OptimizerConfig(**config_dict['optimizer']),
            training=TrainingConfig(**config_dict['training'])
        )
    return config

In [7]:
# Load training data
data_root_dir = "/mnt/data/shared/lc_ms_foundation/"
# data_root_dir = "/mnt/data/shared/lc_ms_foundation/"
dset_name = "abele_data"
data_dir = os.path.join(data_root_dir, dset_name, "mzml")

mzml_files = os.listdir(data_dir)
print("Total N files:", len(mzml_files))

Total N files: 1334


In [8]:
selected_genuses = [
    "Pseudomonas", 
    # "Staphylococcus", 
    "Bacillus", 
    "Escherichia", 
    "Enterococcus",
    "Lactococcus",
    # "Serratia",
    # "Acinetobacter",
]

In [9]:
meta_df = pl.read_csv(os.path.join(
    data_root_dir, 
    dset_name,
    "all_abele_metadata.csv"
))

meta_df = meta_df.rename({
    "characteristics[organism]": "organism",
    "comment[data file]": "data_file"
})

meta_df = meta_df.filter(meta_df["genus"].is_in(selected_genuses))
meta_df

organism,genus,data_file
str,str,str
"""Bacillus cereus""","""Bacillus""","""BBM_429_P110_31_MIA_007"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_437_P110_31_MIA_036"""
"""Bacillus pumilus""","""Bacillus""","""BBM_429_P110_31_MIA_023"""
"""Bacillus subtilis""","""Bacillus""","""BBM_429_P110_31_MIA_024"""
"""Pseudomonas fluorescens""","""Pseudomonas""","""BBM_441_P110_31_MIA_015"""
…,…,…
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_095"""
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_096"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_750_P110_38_MIA_001"""


In [10]:
meta_df["genus"].value_counts()

genus,count
str,u32
"""Enterococcus""",42
"""Bacillus""",109
"""Lactococcus""",21
"""Escherichia""",51
"""Pseudomonas""",312


In [11]:
# Pseudomonas, Bacillus, Escherichia are used for petraining
# and Enterococcus and Lactococcus are split between train and val for downstream eval
# - group by organism, put all files of the same organism in one split

In [12]:
# Use our "default" parameters from config to load data

# TODO: do we maybe need separate configs for different types of experiments?
# (e.g. MS2, MS1, online eval, retrain eval, run, ... ?)
# TODO: would be nice to fix some convenient naming convention for experiments

# config = load_config("../config.yaml")
config = load_config(PROJECT_ROOT / "config.yaml")
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=1000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [13]:
meta_df["genus"].value_counts()

genus,count
str,u32
"""Bacillus""",109
"""Pseudomonas""",312
"""Enterococcus""",42
"""Escherichia""",51
"""Lactococcus""",21


In [14]:
genus_class = {
    "Enterococcus": 0,
    "Lactococcus": 1,
    #
    "Pseudomonas": 2,
    "Bacillus": 3,
    "Escherichia": 4,
}
meta_df = meta_df.with_columns((pl.col("data_file") + ".mzML").alias("peak_file"))
meta_df = meta_df.with_columns(
    (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))
)

display(meta_df["genus_class"].value_counts())
meta_df

/tmp/ipykernel_41649/2649999018.py:11: DeprecationWarning: the `return_dtype` parameter for `replace` is deprecated. Use `replace_strict` instead to set a return data type while replacing values.
(Deprecated in version 1.0.0)
  (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))


genus_class,count
i64,u32
1,21
2,312
0,42
3,109
4,51


organism,genus,data_file,peak_file,genus_class
str,str,str,str,i64
"""Bacillus cereus""","""Bacillus""","""BBM_429_P110_31_MIA_007""","""BBM_429_P110_31_MIA_007.mzML""",3
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_437_P110_31_MIA_036""","""BBM_437_P110_31_MIA_036.mzML""",2
"""Bacillus pumilus""","""Bacillus""","""BBM_429_P110_31_MIA_023""","""BBM_429_P110_31_MIA_023.mzML""",3
"""Bacillus subtilis""","""Bacillus""","""BBM_429_P110_31_MIA_024""","""BBM_429_P110_31_MIA_024.mzML""",3
"""Pseudomonas fluorescens""","""Pseudomonas""","""BBM_441_P110_31_MIA_015""","""BBM_441_P110_31_MIA_015.mzML""",2
…,…,…,…,…
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_095""","""BBM_749_P110_38_MIA_095.mzML""",4
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_096""","""BBM_749_P110_38_MIA_096.mzML""",4
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_750_P110_38_MIA_001""","""BBM_750_P110_38_MIA_001.mzML""",2


In [15]:
# data_dir = "/mnt/data/mpominova/lcms_foundation_data/abele_data"
data_dir = "/mnt/data2/mpominova/lcms_foundation_data/abele_data"
len(os.listdir(data_dir)), os.listdir(data_dir)[:2]

(722, ['BBM_429_P110_31_MIA_026.parquet', 'BBM_429_P110_31_MIA_007.parquet'])

In [16]:
mzml_files = [
    f + ".parquet" for f in meta_df["data_file"].to_list()
    if f + ".parquet" in os.listdir(data_dir)
]
len(mzml_files), mzml_files[:5]

(535,
 ['BBM_429_P110_31_MIA_007.parquet',
  'BBM_437_P110_31_MIA_036.parquet',
  'BBM_429_P110_31_MIA_023.parquet',
  'BBM_429_P110_31_MIA_024.parquet',
  'BBM_441_P110_31_MIA_015.parquet'])

In [17]:
# how do we load only the files we need for the selected genuses?
dfs = {
    mzml_file: pl.read_parquet(os.path.join(data_dir, mzml_file))
    for mzml_file in tqdm(mzml_files)
}
len(dfs)

100%|██████████| 535/535 [00:19<00:00, 27.84it/s]


535

In [18]:
train_genuses = ['Pseudomonas', 'Bacillus', 'Escherichia']
probe_genuses = ["Enterococcus", "Lactococcus"]

In [19]:
g = meta_df.select(["organism", "genus_class"]).group_by("organism")
sorted_organisms = g.first().sort(by=["genus_class", "organism"])["genus_class", "organism"]
sorted_organisms

genus_class,organism
i64,str
0,"""Enterococcus avium"""
0,"""Enterococcus casseliflavus"""
0,"""Enterococcus cecorum"""
0,"""Enterococcus columbae"""
0,"""Enterococcus dispar"""
…,…
3,"""Bacillus sonorensis"""
3,"""Bacillus subtilis"""
3,"""Bacillus thuringiensis"""


In [20]:
train_organisms = sorted_organisms.filter(
    sorted_organisms["genus_class"] >= 2
)["organism"].to_list()

val_organisms = sorted_organisms.filter(
    sorted_organisms["genus_class"].is_in([0, 1])
)["organism"].to_list()

probe_train_organisms = val_organisms[::2]
probe_val_organisms = [
    organism for organism in val_organisms 
    if organism not in probe_train_organisms
]

len(train_organisms), len(probe_train_organisms), len(probe_val_organisms)

(54, 10, 10)

In [21]:
meta_df.filter(
    meta_df["organism"].is_in(probe_train_organisms)
)["genus"].value_counts().sort(by="genus")

genus,count
str,u32
"""Enterococcus""",21
"""Lactococcus""",12


In [22]:
meta_df.filter(
    meta_df["organism"].is_in(probe_val_organisms)
)["genus"].value_counts().sort(by="genus")

genus,count
str,u32
"""Enterococcus""",21
"""Lactococcus""",9


In [23]:
display(sorted_organisms.filter(
    sorted_organisms["organism"].is_in(probe_train_organisms)
))
display(sorted_organisms.filter(
    sorted_organisms["organism"].is_in(probe_val_organisms)
))

genus_class,organism
i64,str
0,"""Enterococcus avium"""
0,"""Enterococcus cecorum"""
0,"""Enterococcus dispar"""
0,"""Enterococcus faecalis"""
0,"""Enterococcus hirae"""
0,"""Enterococcus mundtii"""
0,"""Enterococcus saccharolyticus"""
1,"""Lactococcus cremoris"""
1,"""Lactococcus lactis"""


genus_class,organism
i64,str
0,"""Enterococcus casseliflavus"""
0,"""Enterococcus columbae"""
0,"""Enterococcus durans"""
0,"""Enterococcus faecium"""
0,"""Enterococcus malodoratus"""
0,"""Enterococcus raffinosus"""
0,"""Enterococcus sulfureus"""
1,"""Lactococcus garvieae"""
1,"""Lactococcus piscium"""


In [24]:
organism2split = {}
for organism in train_organisms:
    organism2split[organism] = "train"
for organism in probe_train_organisms:
    organism2split[organism] = "probe_train"
for organism in probe_val_organisms:
    organism2split[organism] = "probe_val"
    
meta_df = meta_df.with_columns(
    pl.col("organism").replace(organism2split).alias("split")
)
meta_df

organism,genus,data_file,peak_file,genus_class,split
str,str,str,str,i64,str
"""Bacillus cereus""","""Bacillus""","""BBM_429_P110_31_MIA_007""","""BBM_429_P110_31_MIA_007.mzML""",3,"""train"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_437_P110_31_MIA_036""","""BBM_437_P110_31_MIA_036.mzML""",2,"""train"""
"""Bacillus pumilus""","""Bacillus""","""BBM_429_P110_31_MIA_023""","""BBM_429_P110_31_MIA_023.mzML""",3,"""train"""
"""Bacillus subtilis""","""Bacillus""","""BBM_429_P110_31_MIA_024""","""BBM_429_P110_31_MIA_024.mzML""",3,"""train"""
"""Pseudomonas fluorescens""","""Pseudomonas""","""BBM_441_P110_31_MIA_015""","""BBM_441_P110_31_MIA_015.mzML""",2,"""train"""
…,…,…,…,…,…
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_095""","""BBM_749_P110_38_MIA_095.mzML""",4,"""train"""
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_096""","""BBM_749_P110_38_MIA_096.mzML""",4,"""train"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_750_P110_38_MIA_001""","""BBM_750_P110_38_MIA_001.mzML""",2,"""train"""


In [25]:
# Data for SSL training
split_df = meta_df.filter(pl.col("split") == "train")
print("SSL train - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

# Data for downstream model training
split_df = meta_df.filter(pl.col("split") == "probe_train")
print("Downstream train - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

# Data for downstream model validation
split_df = meta_df.filter(pl.col("split") == "probe_val")
print("Downstream val - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

SSL train - Total: 472


genus_class,count
i64,u32
2,312
3,109
4,51


Downstream train - Total: 33


genus_class,count
i64,u32
0,21
1,12


Downstream val - Total: 30


genus_class,count
i64,u32
0,21
1,9


In [26]:
# train SpectrumDataset
train_df = pl.concat(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(pl.col("split") == "train")["peak_file"].to_list()
    ], 
    how="vertical"
)
train_df = train_df.join(meta_df, on="peak_file", how="left")

# val SpectrumDataset
val_df = pl.concat(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(
            pl.col("split").is_in(["probe_train", "probe_val"])
        )["peak_file"].to_list()
    ], 
    how="vertical"
)
val_df = val_df.join(meta_df, on="peak_file", how="left")

train_df.shape, val_df.shape

((1194023, 8), (207979, 8))

In [27]:
# we had 1734898 train MS1 spectra before (with Staphylococcus)
# so we might want to add some more data of Pseudomonadota if this performs worse

In [28]:
train_stream = SpectrumDataset(
    train_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", train_stream.n_spectra)
train_lance_path = str(train_stream.path)   # this is the key
print("Lance dataset path:", train_lance_path)
train_dataset = LanceMapDataset(train_lance_path, seq_len=config.data.max_num_peaks)

val_stream = SpectrumDataset(
    val_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", val_stream.n_spectra)
val_lance_path = str(val_stream.path)   # this is the key
print("Lance dataset path:", val_lance_path)
val_dataset = LanceMapDataset(val_lance_path, seq_len=config.data.max_num_peaks)

N train spectra 1194023
Lance dataset path: /tmp/tmpsnmdoy4i/5c722a3d-469f-47cd-814f-e9e526665208.lance
N train spectra 207979
Lance dataset path: /tmp/tmpvwqavycr/b76fcb0e-8b61-4c5d-bcf1-8a136a353de1.lance


In [29]:
run_labels = dict(zip(meta_df["peak_file"], meta_df["genus_class"]))

probe_train_dataset = RunDataset(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_train")["peak_file"].to_list()
    ], 
    run_labels=run_labels, 
    seq_len=config.data.max_num_peaks
)
probe_val_dataset = RunDataset(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_val")["peak_file"].to_list()
    ], 
    run_labels=run_labels,
    seq_len=config.data.max_num_peaks
)
len(probe_train_dataset), len(probe_val_dataset)

100%|██████████| 30/30 [00:00<00:00, 90.36it/s]


(33, 30)

In [30]:
def run_collate_fn(rows):
    # rows is a list of dicts, one per spectrum
    keys = rows[0].keys()
    batch = {}
    for key in keys:
        if key in ['mz_array', 'intensity_array']:
            batch[key] = [torch.tensor(r[key]) for r in rows]
        else:
            batch[key] = torch.tensor([r[key] for r in rows])
    return batch

In [31]:
BATCH_SIZE = config.data.batch_size

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False
)
print("SSL n batches (MS1-level):", len(train_loader), len(val_loader))

probe_train_loader = DataLoader(
    probe_train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True,
    collate_fn=run_collate_fn,
)
probe_val_loader = DataLoader(
    probe_val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False,
    collate_fn=run_collate_fn,
)
print("Downstream n batches (run-level):", len(probe_train_loader), len(probe_val_loader))

SSL n batches (MS1-level): 1195 208
Downstream n batches (run-level): 1 1


In [32]:
# before we used FineTuner callback right here
# but now we would want to import it from  eval/callback.py, right? 
# (I don't know whether eval/ should live inside source/ as well)

from eval.callbacks import FineTuner

In [33]:
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=1000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [34]:
model = MS1Encoder(
    d_model=config.model.d_model,
    nhead=config.model.nhead,
    dim_feedforward=config.model.dim_feedforward,
    n_layers=config.model.n_layers,
    dropout=config.model.dropout,
    n_bins=config.model.n_bins,
    bin_mz_min=config.model.bin_mz_min,
    bin_mz_max=config.model.bin_mz_max,
    masked_peaks_fraction=config.model.masked_peaks_fraction,
    lr=config.optimizer.lr,
    warmup_iters=config.optimizer.warmup_iters,
    cosine_schedule_period_iters=config.optimizer.cosine_schedule_period_iters,
)

In [35]:
os.path.abspath(config.training.checkpoint_path)

'/home/mpominova/lcms-foundation-model/notebooks/train_checkpoints'

In [36]:
root_dir = os.path.join(config.training.checkpoint_path, "foundation_model")
os.makedirs(root_dir, exist_ok=True)

logger = L.loggers.TensorBoardLogger(
    os.path.join(root_dir, "lightning_logs"),
    name=config.name,
)

In [37]:
# TODO: number of classes needs to be parametrized (based on number of classes in val subset)
# and validation classes need to be "renamed" (mapped) to always be 0 < C < n_classes
meta_df.filter(meta_df["organism"].is_in(val_organisms))["genus_class"].value_counts()

genus_class,count
i64,u32
0,42
1,21


In [38]:
retrain_finetuner = FineTuner(
    encoder_output_dim=config.model.d_model, 
    num_classes=2,#len(genus_class),
    target_key="label",
    probe_train_loader=probe_train_loader,
    probe_val_loader=probe_val_loader,
    class_weights=None,
    lr=1e-2,
    n_epochs=50,
    min_train_loss=0.2,
)
# checkpoint_callback = L.ModelCheckpoint(every_n_epochs=100, save_top_k=-1, save_last=True)

trainer = L.Trainer(
    logger=logger,
    default_root_dir=root_dir,
    callbacks=[retrain_finetuner], #checkpoint_callback],
    accelerator="gpu", #config.training.accelerator,
    devices=config.training.devices,
    max_epochs=500, #config.training.max_epochs,
    gradient_clip_val=config.training.gradient_clip_val,
    num_sanity_val_steps=2,
    log_every_n_steps=10,
)

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mpominova/.local/share/mamba/envs/lcms-foundat ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [39]:
# Train the model
trainer.fit(
    model, 
    train_loader, 
    val_dataloaders=[val_loader],
    # ckpt_path=ckpt_path, # 
)

You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type                       | Params | Mode 
-----------------------------------------------------------------------------
0 | peak_encoder          | Sequential                 | 131 K  | train
1 | encoder               | SpectrumTransformerEncoder | 3.3 M  | train
2 | head_mz               | Sequential                 | 308 K  | train
3 | loss_mz_bin           | CrossEntropyLoss           | 0      | train
4 | train_accuracy_mz_bin | MulticlassAccuracy         | 0      | train
5 | val_accuracy_mz_bin   | MulticlassAccuracy         | 0      | train
-----------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  2.57it/s]

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


...

Epoch 0: 100%|██████████| 1195/1195 [16:36<00:00,  1.20it/s, v_num=4, train_acc_mz_bin=0.0435]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 208/208 [01:31<00:00,  2.27it/s]

     targets    prob_0    prob_1
0       1.0  0.459203  0.540797
1       0.0  0.455485  0.544515
2       0.0  0.466442  0.533558
3       0.0  0.463281  0.536719
4       0.0  0.457454  0.542546
5       1.0  0.490250  0.509750
6       1.0  0.454662  0.545338
7       0.0  0.446124  0.553876
8       0.0  0.457876  0.542124
9       0.0  0.459792  0.540208
10      1.0  0.467498  0.532502
11      0.0  0.482141  0.517859
12      0.0  0.508946  0.491054
13      0.0  0.487722  0.512278
14      0.0  0.528720  0.471280
15      1.0  0.484352  0.515647
16      1.0  0.494695  0.505305
17      0.0  0.482747  0.517253
18      0.0  0.493062  0.506938
19      0.0  0.510673  0.489327 



Epoch 0: 100%|██████████| 1195/1195 [18:19<00:00,  1.09it/s, v_num=4, train_acc_mz_bin=0.0435, val_acc_mz_

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)




Validation DataLoader 0:  98%|█████████▊| 203/208 [01:26<00:02,  2.35it/s]

     targets    prob_0    prob_1
0       1.0  0.652880  0.347120
1       0.0  0.624135  0.375865
2       0.0  0.680962  0.319038
3       0.0  0.738293  0.261707
4       0.0  0.608293  0.391707
5       1.0  0.514038  0.485962
6       1.0  0.498399  0.501601
7       0.0  0.620988  0.379012
8       0.0  0.653664  0.346336
9       0.0  0.563649  0.436351
10      1.0  0.651613  0.348387
11      0.0  0.789985  0.210015
12      0.0  0.839461  0.160539
13      0.0  0.883419  0.116581
14      0.0  0.667050  0.332950
15      1.0  0.371101  0.628899
16      1.0  0.702376  0.297624
17      0.0  0.768811  0.231189
18      0.0  0.747977  0.252023
19      0.0  0.651947  0.348053 



Epoch 32: 100%|██████████| 1195/1195 [17:56<00:00,  1.11it/s, v_num=4, train_acc_mz_bin=0.121, val_acc_mz_bin=0.112, retrain_val_auc=0.868, retrain_val_ap=0.872]Probe epoch 0 loss: 0.7974  auc: 0.4960  ap: 0.5980
Probe epoch 1 loss: 0.8817  auc: 


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
# Can it be that end_validation_epoch happens before end_train_epoch? why? 

In [ ]:
# - to build learning curves (later)
    # - several experiments with multiple datasets of various sizes
    # - we can either train for a fixed number of epochs, or steps
    # (probably, steps would be better)